# RQ4 (Portfolio Examination)

## From Scratch to Pretrained -- When Does Pretraining Pay Off?
#### *Comparing LSTM, GloVe LSTM, BiLSTM+Attention, BERT Fine-Tuning, and LLM Prompting*

**Due Date:** May 30, 2026
**Team:** NW 2
**Members:** Nils Wagner, Nick Wenzel

---

### Research Question

> *How do a from-scratch LSTM, an LSTM with GloVe embeddings, a BiLSTM with GloVe and attention pooling, a fine-tuned BERT, and a prompted LLM compare on sentiment classification -- and how much does each improvement (representation, architecture, contextual pretraining) contribute?*

### What This Notebook Does

This notebook covers:
1. IMDB data loading with configurable training sizes (50, 200, 1K, 5K, full)
2. LSTM with random embeddings
3. LSTM with frozen GloVe embeddings
4. BiLSTM with frozen GloVe embeddings and attention pooling
5. DistilBERT/BERT fine-tuning with HuggingFace Trainer
6. GPT-style zero-shot and few-shot prompting
7. Accuracy, macro-F1, training time, inference time, plots, result exports, and error analysis

The analysis cells at the end compute the gap decomposition, crossover points, TextCNN comparison, and practical recommendations for the report.


In [19]:
# -- Setup ---------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score
import re
import time

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f"Using device: {device}")

# HuggingFace imports
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, pipeline
from datasets import load_dataset
import evaluate

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print("Setup complete.")


Using device: cuda
Setup complete.


---
## Part 1: Data Loading with Configurable Subset Sizes

The key variable: **how many labeled training examples are available?**
We test with 50, 200, 1,000, 5,000, and the full 25,000.

In [20]:
# -- Load IMDB dataset ---------------------------------------------------
dataset = load_dataset("imdb")
print(f"Full dataset: Train {len(dataset['train']):,}, Test {len(dataset['test']):,}")

def get_subset(dataset, n_train, seed=42):
    """Sample n_train balanced examples from the training set."""
    if n_train >= len(dataset['train']):
        return dataset['train']
    pos = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 1]
    neg = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 0]
    rng = np.random.RandomState(seed)
    pos_sample = rng.choice(pos, n_train // 2, replace=False)
    neg_sample = rng.choice(neg, n_train // 2, replace=False)
    indices = np.concatenate([pos_sample, neg_sample])
    rng.shuffle(indices)
    return dataset['train'].select(indices.tolist())

def get_balanced_split_subset(split, n_examples=None, seed=42):
    """Return a random balanced subset from a HuggingFace split.

    If n_examples is None, the full split is returned. This is used for
    optional reduced evaluations so the report can state exactly how
    the evaluation sample was chosen.
    """
    if n_examples is None or n_examples >= len(split):
        return split
    if n_examples % 2 != 0:
        raise ValueError("n_examples must be even for a balanced subset.")

    pos = [i for i, ex in enumerate(split) if ex['label'] == 1]
    neg = [i for i, ex in enumerate(split) if ex['label'] == 0]
    per_class = n_examples // 2
    if per_class > min(len(pos), len(neg)):
        raise ValueError("Requested subset is larger than the smallest class.")

    rng = np.random.RandomState(seed)
    indices = np.concatenate([
        rng.choice(pos, per_class, replace=False),
        rng.choice(neg, per_class, replace=False),
    ])
    rng.shuffle(indices)
    return split.select(indices.tolist())

# BERT uses the full IMDB test split by default. Set to an even number such
# as 2000 if runtime requires a documented random, balanced test subset.
BERT_EVAL_MAX_EXAMPLES = None
BERT_EVAL_SEED = 2026

# Prompting also uses the full IMDB test split by default, same as all other experiments.
# Set to an even number only if runtime requires a documented random balanced subset.
PROMPT_EVAL_N = None

# Test balanced sampling helper
for n in [50, 200, 1000, 5000]:
    subset = get_subset(dataset, n)
    labels = [ex['label'] for ex in subset]
    print(f"  n={n:5d}: {len(subset)} examples, {sum(labels)} pos, {len(labels)-sum(labels)} neg")

TRAIN_SIZES = [50, 200, 1000, 5000, 25000]
N_SEEDS = 3
print(f"\nExperiment plan: {len(TRAIN_SIZES)} sizes x {N_SEEDS} seeds = {len(TRAIN_SIZES)*N_SEEDS} runs per approach")


Full dataset: Train 25,000, Test 25,000
  n=   50: 50 examples, 25 pos, 25 neg
  n=  200: 200 examples, 100 pos, 100 neg
  n= 1000: 1000 examples, 500 pos, 500 neg
  n= 5000: 5000 examples, 2500 pos, 2500 neg

Experiment plan: 5 sizes x 3 seeds = 15 runs per approach


---
## Part 2: Model 1 — LSTM (From Scratch)

Reuses the LSTM architecture from Week 6 / RQ3. No pretrained embeddings — learns everything from the labeled training data.

In [21]:
# ── Text pipeline for LSTM ─────────────────────────────────────
def tokenize(text):
    text = text.lower()
    text = re.sub(r'<br\s*/?>', ' ', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text.split()

def build_vocab(texts, max_vocab=20000, min_freq=2):
    counter = Counter()
    for text in texts:
        counter.update(tokenize(text))
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for word, count in counter.most_common(max_vocab - 2):
        if count >= min_freq:
            vocab[word] = len(vocab)
    return vocab

def encode(text, vocab, max_len=200):
    tokens = tokenize(text)[:max_len]
    ids = [vocab.get(t, vocab['<UNK>']) for t in tokens]
    ids = ids + [0] * (max_len - len(ids))
    return ids

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=200):
        self.encodings = [encode(t, vocab, max_len) for t in texts]
        self.labels = labels
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return (torch.tensor(self.encodings[idx], dtype=torch.long),
                torch.tensor(self.labels[idx], dtype=torch.long))

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        x = self.embedding(x)
        _, (h_n, _) = self.lstm(x)
        return self.fc(self.dropout(h_n.squeeze(0)))

print("LSTM pipeline defined.")

LSTM pipeline defined.


In [22]:
# -- Train and evaluate LSTM --------------------------------------------
def evaluate_torch_classifier(model, test_dataset, batch_size=128):
    """Evaluate a torch classifier and return accuracy, macro-F1, and timing."""
    model.eval()
    loader = DataLoader(test_dataset, batch_size=batch_size)
    preds, labels = [], []
    t0 = time.time()
    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device)
            logits = model(bx)
            preds.extend(logits.argmax(1).cpu().numpy().tolist())
            labels.extend(by.numpy().tolist())
    elapsed = time.time() - t0
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average="macro", labels=[0, 1], zero_division=0),
        'inference_time_per_example': elapsed / len(labels),
        'n_eval': len(labels),
    }

def train_lstm(train_subset, test_dataset, vocab, epochs=10, lr=1e-3):
    """Train LSTM on a subset and evaluate on the provided test dataset."""
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]

    train_ds = SentimentDataset(train_texts, train_labels, vocab)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

    model = LSTMClassifier(len(vocab)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    train_time = time.time() - t0

    metrics = evaluate_torch_classifier(model, test_dataset)
    metrics['train_time'] = train_time
    return metrics

# Build vocab from FULL training set so it is consistent across subset sizes.
all_train_texts = [ex['text'] for ex in dataset['train']]
vocab = build_vocab(all_train_texts)
print(f"Vocabulary: {len(vocab):,} words")

# Full IMDB test set for LSTM-family evaluations.
test_texts = [ex['text'] for ex in dataset['test']]
test_labels = [ex['label'] for ex in dataset['test']]
test_ds = SentimentDataset(test_texts, test_labels, vocab)
print(f"LSTM-family test set: {len(test_ds):,} examples")


Vocabulary: 20,000 words
LSTM-family test set: 25,000 examples


---
## Part 2b: Model 2 — LSTM with GloVe Embeddings (Static Pretraining)

Same LSTM architecture, but with pretrained GloVe word vectors instead of random embeddings. This tests: **how much of the LSTM's weakness in RQ3 was due to learning embeddings from scratch?**

In [23]:
# ── Load GloVe embeddings ──────────────────────────────────────
import gensim.downloader as api

print("Loading GloVe embeddings (this may take a moment on first run)...")
try:
    glove = api.load("glove-wiki-gigaword-100")  # 100d GloVe vectors
    GLOVE_DIM = 100
    print(f"Loaded GloVe: {len(glove.key_to_index):,} words, {GLOVE_DIM} dimensions")
except Exception as e:
    print(f"Could not load GloVe via gensim: {e}")
    print("Falling back to random embeddings for the GloVe condition.")
    glove = None
    GLOVE_DIM = 100

Loading GloVe embeddings (this may take a moment on first run)...
Loaded GloVe: 400,000 words, 100 dimensions


In [24]:
# -- Build GloVe embedding matrix for our vocabulary --------------------
def build_glove_matrix(vocab, glove_model, embed_dim=100):
    """Create an embedding matrix from GloVe for our vocabulary."""
    matrix = np.random.randn(len(vocab), embed_dim) * 0.01
    found, total = 0, 0
    for word, idx in vocab.items():
        total += 1
        if glove_model is not None and word in glove_model:
            matrix[idx] = glove_model[word]
            found += 1
    print(f"GloVe coverage: {found}/{total} words ({found/total:.1%})")
    return torch.tensor(matrix, dtype=torch.float32)

glove_matrix = build_glove_matrix(vocab, glove)

def train_lstm_glove(train_subset, test_dataset, vocab, glove_matrix,
                     epochs=10, lr=1e-3):
    """Train LSTM with frozen GloVe embeddings."""
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]

    train_ds = SentimentDataset(train_texts, train_labels, vocab)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

    model = LSTMClassifier(len(vocab), embed_dim=GLOVE_DIM).to(device)
    model.embedding.weight.data.copy_(glove_matrix)
    model.embedding.weight.requires_grad = False

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.CrossEntropyLoss()

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    train_time = time.time() - t0

    metrics = evaluate_torch_classifier(model, test_dataset)
    metrics['train_time'] = train_time
    return metrics

print("LSTM + GloVe pipeline defined.")


GloVe coverage: 19865/20000 words (99.3%)
LSTM + GloVe pipeline defined.


---
## Part 2c: Model 3 — BiLSTM + GloVe + Attention Pooling

Same GloVe embeddings as Model 2, but with a **bidirectional LSTM** and **attention pooling** over all hidden states. This isolates the effect of better architecture and aggregation.

In [25]:
# -- BiLSTM with attention pooling --------------------------------------
class AttentionPool(nn.Module):
    """Learned attention pooling over LSTM hidden states."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, lstm_output):
        scores = self.attn(lstm_output).squeeze(-1)
        weights = F.softmax(scores, dim=1)
        return (weights.unsqueeze(-1) * lstm_output).sum(dim=1)

class BiLSTMAttnClassifier(nn.Module):
    """BiLSTM with GloVe embeddings and attention pooling."""
    def __init__(self, vocab_size, embed_dim=100, hidden_dim=128, num_classes=2,
                 dropout=0.5, pretrained_embeddings=None, freeze_emb=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(pretrained_embeddings)
        if freeze_emb:
            self.embedding.weight.requires_grad = False
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.attention = AttentionPool(hidden_dim * 2)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        pooled = self.attention(lstm_out)
        return self.fc(self.dropout(pooled))

def train_bilstm_glove(train_subset, test_dataset, vocab, glove_matrix,
                       epochs=10, lr=1e-3):
    """Train BiLSTM+attention with frozen GloVe embeddings."""
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]
    train_ds = SentimentDataset(train_texts, train_labels, vocab)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

    model = BiLSTMAttnClassifier(
        len(vocab), embed_dim=GLOVE_DIM,
        pretrained_embeddings=glove_matrix, freeze_emb=True
    ).to(device)

    optimizer = torch.optim.Adam(
        [p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.CrossEntropyLoss()

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
    train_time = time.time() - t0

    metrics = evaluate_torch_classifier(model, test_dataset)
    metrics['train_time'] = train_time
    return metrics

print("BiLSTM + GloVe + AttentionPool defined.")
print("  Compared to Model 2: adds bidirectionality + attention pooling")
print("  Compared to Model 1: adds GloVe + bidirectionality + attention pooling")


BiLSTM + GloVe + AttentionPool defined.
  Compared to Model 2: adds bidirectionality + attention pooling
  Compared to Model 1: adds GloVe + bidirectionality + attention pooling


---
## Part 3: Model 4 — BERT Fine-Tuning

Fine-tune DistilBERT (faster than full BERT, 97% of the quality) using HuggingFace.

In [26]:
# -- BERT fine-tuning pipeline ------------------------------------------
BERT_MODEL = "distilbert-base-uncased"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1_score(labels, preds, average="macro", labels=[0, 1], zero_division=0),
    }

class SimpleDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings['labels'])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encodings.items()}

def train_bert(train_subset, n_train, epochs=3, lr=2e-5):
    """Fine-tune BERT and evaluate on the full test set by default."""
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]

    train_enc = bert_tokenizer(train_texts, truncation=True, padding="max_length",
                               max_length=256, return_tensors="pt")
    train_enc['labels'] = torch.tensor(train_labels)

    test_sample = get_balanced_split_subset(
        dataset['test'], n_examples=BERT_EVAL_MAX_EXAMPLES, seed=BERT_EVAL_SEED
    )
    test_texts_b = [ex['text'] for ex in test_sample]
    test_labels_b = [ex['label'] for ex in test_sample]
    test_enc = bert_tokenizer(test_texts_b, truncation=True, padding="max_length",
                              max_length=256, return_tensors="pt")
    test_enc['labels'] = torch.tensor(test_labels_b)

    model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2)

    args = TrainingArguments(
        output_dir=f"./tmp_rq4_{n_train}",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=lr,
        weight_decay=0.01,
        eval_strategy="no",
        save_strategy="no",
        logging_steps=9999,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(model=model, args=args,
                      train_dataset=SimpleDataset(train_enc),
                      eval_dataset=SimpleDataset(test_enc),
                      compute_metrics=compute_metrics)

    t0 = time.time()
    trainer.train()
    train_time = time.time() - t0

    eval_t0 = time.time()
    eval_results = trainer.evaluate()
    eval_time = time.time() - eval_t0

    return {
        'accuracy': eval_results['eval_accuracy'],
        'macro_f1': eval_results['eval_macro_f1'],
        'train_time': train_time,
        'inference_time_per_example': eval_time / len(test_sample),
        'n_eval': len(test_sample),
    }

print("BERT fine-tuning pipeline defined.")
print(f"Using: {BERT_MODEL}")
print("BERT evaluation: full IMDB test split unless BERT_EVAL_MAX_EXAMPLES is set.")


BERT fine-tuning pipeline defined.
Using: distilbert-base-uncased
BERT evaluation: full IMDB test split unless BERT_EVAL_MAX_EXAMPLES is set.


---
## Part 4: Model 5 — LLM Prompting

Classify sentiment using prompting — no training, no weight updates. The model classifies based on the prompt alone.

In [ ]:
# -- Prompting pipeline --------------------------------------------------
# Use a text-generation model for prompting. GPT-2 is a weak local baseline.
PROMPT_MODEL = "gpt2"
PROMPT_BATCH_SIZE = 16

prompt_generator = pipeline("text-generation", model=PROMPT_MODEL,
                            device=device, max_new_tokens=5)

def build_prompt(text, examples=None):
    """Build the zero-shot or few-shot sentiment prompt."""
    if examples:
        prompt_parts = ["Classify sentiment as Positive or Negative.\n"]
        for ex_text, ex_label in examples:
            label_str = "Positive" if ex_label == 1 else "Negative"
            prompt_parts.append(f'Review: "{ex_text[:100]}" -> {label_str}')
        prompt_parts.append(f'Review: "{text[:200]}" ->')
        return "\n".join(prompt_parts)
    return f'Classify the sentiment as Positive or Negative.\nReview: "{text[:200]}"\nSentiment:'

def parse_prompt_output(generated):
    """Parse the generated continuation into a sentiment label."""
    generated = generated.strip().lower()
    if 'positive' in generated[:20]:
        return 1
    if 'negative' in generated[:20]:
        return 0
    return -1

def classify_with_prompt(text, examples=None):
    """Classify one review using prompting."""
    prompt = build_prompt(text, examples)
    try:
        result = prompt_generator(prompt, pad_token_id=prompt_generator.tokenizer.eos_token_id)[0]
        generated = result['generated_text'][len(prompt):]
        return parse_prompt_output(generated)
    except Exception:
        return -1

def generate_prompt_predictions(prompts, batch_size=PROMPT_BATCH_SIZE, progress_every=5000):
    """Generate predictions for a list of prompts, batching for full-test evaluation."""
    preds = []
    t0 = time.time()
    for start in range(0, len(prompts), batch_size):
        batch = prompts[start:start + batch_size]
        try:
            outputs = prompt_generator(
                batch,
                batch_size=batch_size,
                pad_token_id=prompt_generator.tokenizer.eos_token_id,
            )
            for prompt, output in zip(batch, outputs):
                item = output[0] if isinstance(output, list) else output
                generated_text = item.get('generated_text', '')
                continuation = generated_text[len(prompt):] if generated_text.startswith(prompt) else generated_text
                preds.append(parse_prompt_output(continuation))
        except Exception:
            for prompt in batch:
                try:
                    output = prompt_generator(prompt, pad_token_id=prompt_generator.tokenizer.eos_token_id)[0]
                    generated_text = output.get('generated_text', '')
                    continuation = generated_text[len(prompt):] if generated_text.startswith(prompt) else generated_text
                    preds.append(parse_prompt_output(continuation))
                except Exception:
                    preds.append(-1)
        if progress_every and (start + len(batch)) % progress_every == 0:
            print(f"    prompted {start + len(batch):,}/{len(prompts):,} examples")
    return preds, time.time() - t0

def evaluate_prompting(test_split, examples=None, max_test=PROMPT_EVAL_N, seed=42):
    """Evaluate prompting on the full test split by default.

    accuracy counts unparsed outputs as wrong. accuracy_parsed reports the
    parsed-only score for transparency.
    """
    eval_sample = get_balanced_split_subset(test_split, n_examples=max_test, seed=seed)
    eval_texts = [ex['text'] for ex in eval_sample]
    eval_labels = [ex['label'] for ex in eval_sample]
    prompts = [build_prompt(text, examples) for text in eval_texts]

    preds, elapsed = generate_prompt_predictions(prompts)

    parsed_idx = [i for i, pred in enumerate(preds) if pred != -1]
    parsed_labels = [eval_labels[i] for i in parsed_idx]
    parsed_preds = [preds[i] for i in parsed_idx]
    unparsed = len(preds) - len(parsed_preds)

    accuracy_all = accuracy_score(eval_labels, preds)
    macro_f1_all = f1_score(eval_labels, preds, average="macro", labels=[0, 1], zero_division=0)
    accuracy_parsed = accuracy_score(parsed_labels, parsed_preds) if parsed_preds else 0.0
    macro_f1_parsed = (
        f1_score(parsed_labels, parsed_preds, average="macro", labels=[0, 1], zero_division=0)
        if parsed_preds else 0.0
    )

    return {
        'accuracy': accuracy_all,
        'accuracy_parsed': accuracy_parsed,
        'macro_f1': macro_f1_all,
        'macro_f1_parsed': macro_f1_parsed,
        'parse_rate': len(parsed_preds) / len(preds),
        'unparsed': unparsed,
        'train_time': 0.0,
        'inference_time_per_example': elapsed / len(preds),
        'n_eval': len(preds),
    }

print(f"Prompting pipeline defined. Using: {PROMPT_MODEL}")
print("Prompting evaluation uses the full IMDB test split by default and stores parse diagnostics.")


---
## Part 5: Run All Experiments

The core experiment loop: 3 approaches × 5+ dataset sizes × 3 seeds.

**Warning:** This takes a while! BERT fine-tuning is the bottleneck.
On CPU: ~1-2 hours total. On GPU: ~20-30 minutes.

In [28]:
# -- Experiment loop -----------------------------------------------------
results = {}

def summarize_runs(run_metrics):
    """Aggregate per-seed metrics and keep the raw seed rows."""
    fields = [
        'accuracy', 'macro_f1', 'train_time', 'inference_time_per_example',
        'accuracy_parsed', 'macro_f1_parsed', 'parse_rate', 'unparsed', 'n_eval',
    ]
    summary = {'runs': run_metrics}
    for field in fields:
        values = [row[field] for row in run_metrics if field in row and row[field] is not None]
        if values:
            summary[f'{field}_mean'] = float(np.mean(values))
            summary[f'{field}_std'] = float(np.std(values))
    summary['mean'] = summary.get('accuracy_mean', np.nan)
    summary['std'] = summary.get('accuracy_std', np.nan)
    return summary

def print_run(model_name, n_train, seed, metrics):
    print(
        f"  n={n_train:5d}, seed={seed}: "
        f"acc={metrics['accuracy']:.4f}, "
        f"macro_f1={metrics['macro_f1']:.4f}, "
        f"train={metrics['train_time']:.1f}s, "
        f"infer/ex={metrics['inference_time_per_example']:.6f}s"
    )

# -- LSTM experiments ---------------------------------------------------
print("=" * 60)
print("LSTM EXPERIMENTS")
print("=" * 60)

for n_train in TRAIN_SIZES:
    run_metrics = []
    for seed in range(N_SEEDS):
        subset = get_subset(dataset, n_train, seed=42 + seed)
        metrics = train_lstm(subset, test_ds, vocab, epochs=10)
        metrics.update({'seed': seed, 'model': 'LSTM', 'N': n_train})
        run_metrics.append(metrics)
        print_run('LSTM', n_train, seed, metrics)
    results[('LSTM', n_train)] = summarize_runs(run_metrics)
    r = results[('LSTM', n_train)]
    print(f"  -> LSTM n={n_train}: acc={r['accuracy_mean']:.4f} +/- {r['accuracy_std']:.4f}, macro_f1={r['macro_f1_mean']:.4f}")
    print()


LSTM EXPERIMENTS
  n=   50, seed=0: acc=0.5030, macro_f1=0.4766, train=0.0s, infer/ex=0.000029s
  n=   50, seed=1: acc=0.5048, macro_f1=0.4061, train=0.0s, infer/ex=0.000028s
  n=   50, seed=2: acc=0.5104, macro_f1=0.4845, train=0.0s, infer/ex=0.000028s
  -> LSTM n=50: acc=0.5061 +/- 0.0031, macro_f1=0.4557

  n=  200, seed=0: acc=0.5021, macro_f1=0.4499, train=0.1s, infer/ex=0.000029s
  n=  200, seed=1: acc=0.5053, macro_f1=0.4338, train=0.1s, infer/ex=0.000028s
  n=  200, seed=2: acc=0.5054, macro_f1=0.4847, train=0.1s, infer/ex=0.000028s
  -> LSTM n=200: acc=0.5043 +/- 0.0016, macro_f1=0.4561

  n= 1000, seed=0: acc=0.5027, macro_f1=0.4791, train=0.6s, infer/ex=0.000028s
  n= 1000, seed=1: acc=0.5114, macro_f1=0.4763, train=0.6s, infer/ex=0.000028s
  n= 1000, seed=2: acc=0.5116, macro_f1=0.4543, train=0.6s, infer/ex=0.000028s
  -> LSTM n=1000: acc=0.5086 +/- 0.0041, macro_f1=0.4699

  n= 5000, seed=0: acc=0.6021, macro_f1=0.6006, train=2.8s, infer/ex=0.000028s
  n= 5000, seed=1: acc

In [29]:
# -- LSTM + GloVe experiments -------------------------------------------
print("=" * 60)
print("LSTM + GloVe EXPERIMENTS")
print("=" * 60)

for n_train in TRAIN_SIZES:
    run_metrics = []
    for seed in range(N_SEEDS):
        subset = get_subset(dataset, n_train, seed=42 + seed)
        metrics = train_lstm_glove(subset, test_ds, vocab, glove_matrix, epochs=10)
        metrics.update({'seed': seed, 'model': 'LSTM+GloVe', 'N': n_train})
        run_metrics.append(metrics)
        print_run('LSTM+GloVe', n_train, seed, metrics)
    results[('LSTM+GloVe', n_train)] = summarize_runs(run_metrics)
    r = results[('LSTM+GloVe', n_train)]
    print(f"  -> LSTM+GloVe n={n_train}: acc={r['accuracy_mean']:.4f} +/- {r['accuracy_std']:.4f}, macro_f1={r['macro_f1_mean']:.4f}")
    print()


LSTM + GloVe EXPERIMENTS
  n=   50, seed=0: acc=0.5059, macro_f1=0.5036, train=0.0s, infer/ex=0.000028s
  n=   50, seed=1: acc=0.4946, macro_f1=0.4920, train=0.0s, infer/ex=0.000028s
  n=   50, seed=2: acc=0.5066, macro_f1=0.5046, train=0.0s, infer/ex=0.000028s
  -> LSTM+GloVe n=50: acc=0.5023 +/- 0.0055, macro_f1=0.5001

  n=  200, seed=0: acc=0.5124, macro_f1=0.5042, train=0.1s, infer/ex=0.000028s
  n=  200, seed=1: acc=0.4978, macro_f1=0.4016, train=0.1s, infer/ex=0.000028s
  n=  200, seed=2: acc=0.5139, macro_f1=0.5067, train=0.1s, infer/ex=0.000028s
  -> LSTM+GloVe n=200: acc=0.5081 +/- 0.0072, macro_f1=0.4708

  n= 1000, seed=0: acc=0.5379, macro_f1=0.5285, train=0.5s, infer/ex=0.000029s
  n= 1000, seed=1: acc=0.5104, macro_f1=0.4251, train=0.5s, infer/ex=0.000028s
  n= 1000, seed=2: acc=0.5148, macro_f1=0.4653, train=0.5s, infer/ex=0.000028s
  -> LSTM+GloVe n=1000: acc=0.5210 +/- 0.0120, macro_f1=0.4730

  n= 5000, seed=0: acc=0.5394, macro_f1=0.4776, train=2.4s, infer/ex=0.0000

In [30]:
# -- BiLSTM + GloVe + Attention experiments -----------------------------
print("=" * 60)
print("BiLSTM + GloVe + ATTENTION EXPERIMENTS")
print("=" * 60)

for n_train in TRAIN_SIZES:
    run_metrics = []
    for seed in range(N_SEEDS):
        subset = get_subset(dataset, n_train, seed=42 + seed)
        metrics = train_bilstm_glove(subset, test_ds, vocab, glove_matrix, epochs=10)
        metrics.update({'seed': seed, 'model': 'BiLSTM+Attn', 'N': n_train})
        run_metrics.append(metrics)
        print_run('BiLSTM+Attn', n_train, seed, metrics)
    results[('BiLSTM+Attn', n_train)] = summarize_runs(run_metrics)
    r = results[('BiLSTM+Attn', n_train)]
    print(f"  -> BiLSTM+Attn n={n_train}: acc={r['accuracy_mean']:.4f} +/- {r['accuracy_std']:.4f}, macro_f1={r['macro_f1_mean']:.4f}")
    print()


BiLSTM + GloVe + ATTENTION EXPERIMENTS
  n=   50, seed=0: acc=0.5592, macro_f1=0.5195, train=0.1s, infer/ex=0.000045s
  n=   50, seed=1: acc=0.5548, macro_f1=0.5499, train=0.1s, infer/ex=0.000045s
  n=   50, seed=2: acc=0.5236, macro_f1=0.4506, train=0.1s, infer/ex=0.000045s
  -> BiLSTM+Attn n=50: acc=0.5459 +/- 0.0159, macro_f1=0.5067

  n=  200, seed=0: acc=0.6120, macro_f1=0.5965, train=0.2s, infer/ex=0.000045s
  n=  200, seed=1: acc=0.6589, macro_f1=0.6570, train=0.2s, infer/ex=0.000045s
  n=  200, seed=2: acc=0.6156, macro_f1=0.6031, train=0.2s, infer/ex=0.000045s
  -> BiLSTM+Attn n=200: acc=0.6288 +/- 0.0213, macro_f1=0.6189

  n= 1000, seed=0: acc=0.7589, macro_f1=0.7582, train=1.1s, infer/ex=0.000045s
  n= 1000, seed=1: acc=0.7429, macro_f1=0.7376, train=1.0s, infer/ex=0.000044s
  n= 1000, seed=2: acc=0.7493, macro_f1=0.7476, train=1.0s, infer/ex=0.000045s
  -> BiLSTM+Attn n=1000: acc=0.7504 +/- 0.0066, macro_f1=0.7478

  n= 5000, seed=0: acc=0.8346, macro_f1=0.8340, train=5.1s

In [31]:
# -- BERT experiments ----------------------------------------------------
print("=" * 60)
print("BERT EXPERIMENTS")
print("=" * 60)

for n_train in TRAIN_SIZES:
    run_metrics = []
    for seed in range(N_SEEDS):
        subset = get_subset(dataset, n_train, seed=42 + seed)
        metrics = train_bert(subset, n_train, epochs=3)
        metrics.update({'seed': seed, 'model': 'BERT', 'N': n_train})
        run_metrics.append(metrics)
        print_run('BERT', n_train, seed, metrics)
    results[('BERT', n_train)] = summarize_runs(run_metrics)
    r = results[('BERT', n_train)]
    print(f"  -> BERT n={n_train}: acc={r['accuracy_mean']:.4f} +/- {r['accuracy_std']:.4f}, macro_f1={r['macro_f1_mean']:.4f}")
    print()


BERT EXPERIMENTS


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Training Loss,Validation Loss,Step,Accuracy,Macro F1
No log,0.687471,12,0.596240,0.575985


...

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss


Training Loss,Validation Loss,Step,Accuracy,Macro F1
No log,0.344109,4689,0.914360,0.914360


  n=25000, seed=2: acc=0.9144, macro_f1=0.9144, train=311.5s, infer/ex=0.000879s
  -> BERT n=25000: acc=0.9145 +/- 0.0001, macro_f1=0.9145



In [ ]:
# -- Prompting experiments ----------------------------------------------
print("=" * 60)
print("PROMPTING EXPERIMENTS")
print("=" * 60)

# Zero-shot has no random demonstrations, so one full-test run is sufficient.
metrics = evaluate_prompting(dataset['test'], max_test=PROMPT_EVAL_N, seed=900)
metrics.update({'seed': 0, 'model': 'Prompt-ZS', 'N': 0})
results[('Prompt-ZS', 0)] = summarize_runs([metrics])
print(
    f"  Zero-shot: acc_all={metrics['accuracy']:.4f}, "
    f"acc_parsed={metrics['accuracy_parsed']:.4f}, "
    f"parse_rate={metrics['parse_rate']:.2%}, unparsed={metrics['unparsed']}"
)

# Few-shot with k examples: repeat across seeds because demonstrations change.
for k in [2, 4, 8]:
    run_metrics = []
    for seed in range(N_SEEDS):
        rng = np.random.RandomState(42 + seed)
        pos = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 1]
        neg = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 0]
        ex_idx = list(rng.choice(pos, k // 2, replace=False)) + list(rng.choice(neg, k // 2, replace=False))
        rng.shuffle(ex_idx)
        examples = [(dataset['train'][i]['text'], dataset['train'][i]['label']) for i in ex_idx]

        metrics = evaluate_prompting(dataset['test'], examples, max_test=PROMPT_EVAL_N, seed=1200 + seed)
        metrics.update({'seed': seed, 'model': f'Prompt-{k}shot', 'N': k})
        run_metrics.append(metrics)
        print(
            f"  Few-shot k={k}, seed={seed}: acc_all={metrics['accuracy']:.4f}, "
            f"acc_parsed={metrics['accuracy_parsed']:.4f}, "
            f"parse_rate={metrics['parse_rate']:.2%}, unparsed={metrics['unparsed']}"
        )

    results[(f'Prompt-{k}shot', k)] = summarize_runs(run_metrics)
    r = results[(f'Prompt-{k}shot', k)]
    print(f"  -> Prompt k={k}: acc_all={r['accuracy_mean']:.4f} +/- {r['accuracy_std']:.4f}, parse_rate={r['parse_rate_mean']:.2%}")

print("\nAll experiments complete!")


---
## Part 6: The Key Plot — Accuracy vs. Dataset Size

In [ ]:
# -- The central figure --------------------------------------------------
from pathlib import Path

fig, ax = plt.subplots(figsize=(12, 7))

lstm_sizes = TRAIN_SIZES

def result_value(key, field='accuracy_mean'):
    if key not in results:
        return np.nan
    row = results[key]
    legacy = {'accuracy_mean': 'mean', 'accuracy_std': 'std'}
    if field in row:
        return row[field]
    if field in legacy and legacy[field] in row:
        return row[legacy[field]]
    return np.nan

lstm_accs = [result_value(('LSTM', n)) for n in lstm_sizes]
lstm_stds = [result_value(('LSTM', n), 'accuracy_std') for n in lstm_sizes]
ax.errorbar(lstm_sizes, lstm_accs, yerr=lstm_stds, fmt='o-', color='#E85D75',
            linewidth=2, markersize=8, capsize=4, label='LSTM (random emb.)')

glove_accs = [result_value(('LSTM+GloVe', n)) for n in lstm_sizes]
glove_stds = [result_value(('LSTM+GloVe', n), 'accuracy_std') for n in lstm_sizes]
ax.errorbar(lstm_sizes, glove_accs, yerr=glove_stds, fmt='^-', color='#E8A838',
            linewidth=2, markersize=8, capsize=4, label='LSTM (GloVe emb.)')

bi_accs = [result_value(('BiLSTM+Attn', n)) for n in lstm_sizes]
bi_stds = [result_value(('BiLSTM+Attn', n), 'accuracy_std') for n in lstm_sizes]
ax.errorbar(lstm_sizes, bi_accs, yerr=bi_stds, fmt='D-', color='#8B5CF6',
            linewidth=2, markersize=8, capsize=4, label='BiLSTM+GloVe+Attn')

bert_accs = [result_value(('BERT', n)) for n in lstm_sizes]
bert_stds = [result_value(('BERT', n), 'accuracy_std') for n in lstm_sizes]
ax.errorbar(lstm_sizes, bert_accs, yerr=bert_stds, fmt='s-', color='#27AE60',
            linewidth=2, markersize=8, capsize=4, label='BERT (fine-tuned)')

prompt_points = [
    (('Prompt-ZS', 0), 0, '#4A90D9', 'Zero-shot prompting'),
    (('Prompt-2shot', 2), 2, '#8B5CF6', '2-shot prompting'),
    (('Prompt-4shot', 4), 4, '#8B5CF6', '4-shot prompting'),
    (('Prompt-8shot', 8), 8, '#8B5CF6', '8-shot prompting'),
]
for key, x, color, label in prompt_points:
    if key in results:
        y = result_value(key)
        yerr = result_value(key, 'accuracy_std')
        ax.axhline(y=y, color=color, linestyle='--' if x == 0 else ':',
                   linewidth=1.5, alpha=0.55, label=label)
        ax.errorbar([x], [y], yerr=[0 if np.isnan(yerr) else yerr], fmt='P',
                    color=color, markersize=9, capsize=4)

ax.set_xscale('symlog', linthresh=10)
ax.set_xlabel("Number of Training Examples / Prompt Demonstrations (symlog scale)", fontsize=13)
ax.set_ylabel("Test Accuracy", fontsize=13)
ax.set_title("RQ4: From Scratch to Pretrained on IMDB", fontweight='bold', fontsize=14)
ax.legend(fontsize=10, loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_ylim(0.0, 1.0)
all_xticks = [0, 2, 4, 8] + TRAIN_SIZES
ax.set_xticks(all_xticks)
ax.set_xticklabels([str(n) for n in all_xticks])

plt.tight_layout()
plt.savefig("rq4_accuracy_vs_size.png", dpi=200, bbox_inches="tight")
Path("rq4_outputs").mkdir(exist_ok=True)
plt.savefig("rq4_outputs/rq4_accuracy_vs_size.png", dpi=200, bbox_inches="tight")
plt.show()

def first_overtake(model_a, model_b):
    """First training size where model_a has higher mean accuracy than model_b."""
    for n in TRAIN_SIZES:
        a = result_value((model_a, n))
        b = result_value((model_b, n))
        if not np.isnan(a) and not np.isnan(b) and a > b:
            return n
    return "not observed"

best_prompt = None
prompt_rows = []
for key in [('Prompt-ZS', 0), ('Prompt-2shot', 2), ('Prompt-4shot', 4), ('Prompt-8shot', 8)]:
    if key in results:
        prompt_rows.append((key[0], result_value(key)))
if prompt_rows:
    best_prompt = max(prompt_rows, key=lambda x: x[1])

print("Crossover notes")
print(f"- BERT overtakes LSTM at n = {first_overtake('BERT', 'LSTM')}")
print(f"- BERT overtakes LSTM+GloVe at n = {first_overtake('BERT', 'LSTM+GloVe')}")
print(f"- BERT overtakes BiLSTM+Attn at n = {first_overtake('BERT', 'BiLSTM+Attn')}")
if best_prompt:
    bert_vs_prompt = "not observed"
    for n in TRAIN_SIZES:
        if result_value(('BERT', n)) > best_prompt[1]:
            bert_vs_prompt = n
            break
    print(f"- Best prompting baseline: {best_prompt[0]} ({best_prompt[1]:.4f})")
    print(f"- BERT overtakes best prompting baseline at n = {bert_vs_prompt}")


### Results Table

In [ ]:
# -- Print and save results table ---------------------------------------
from pathlib import Path
import csv

output_dir = Path("rq4_outputs")
output_dir.mkdir(exist_ok=True)

def metric_or_nan(row, name, fallback=None):
    if row is None:
        return np.nan
    if name in row:
        return row[name]
    if fallback and fallback in row:
        return row[fallback]
    return np.nan

def format_number(value, digits=4):
    try:
        if value is None or np.isnan(value):
            return ""
    except TypeError:
        pass
    return f"{value:.{digits}f}"

def evaluation_status(model, row):
    n_eval = metric_or_nan(row, 'n_eval_mean')
    expected = len(dataset['test'])
    if model.startswith('Prompt') and PROMPT_EVAL_N is None and not np.isnan(n_eval) and int(round(n_eval)) != expected:
        return f"STALE_SUBSET_RUN_EXPECTED_{expected}"
    if not np.isnan(n_eval) and int(round(n_eval)) == expected:
        return "full_test"
    return ""

result_rows = []
for model in ['LSTM', 'LSTM+GloVe', 'BiLSTM+Attn', 'BERT']:
    for n in TRAIN_SIZES:
        r = results.get((model, n))
        if r:
            result_rows.append({
                'model': model,
                'N': n,
                'accuracy_mean': metric_or_nan(r, 'accuracy_mean', 'mean'),
                'accuracy_std': metric_or_nan(r, 'accuracy_std', 'std'),
                'macro_f1_mean': metric_or_nan(r, 'macro_f1_mean'),
                'macro_f1_std': metric_or_nan(r, 'macro_f1_std'),
                'train_time_mean': metric_or_nan(r, 'train_time_mean'),
                'inference_time_per_example': metric_or_nan(r, 'inference_time_per_example_mean'),
                'accuracy_parsed_mean': metric_or_nan(r, 'accuracy_parsed_mean'),
                'parse_rate_mean': metric_or_nan(r, 'parse_rate_mean'),
                'unparsed_mean': metric_or_nan(r, 'unparsed_mean'),
                'n_eval_mean': metric_or_nan(r, 'n_eval_mean'),
                'evaluation_status': evaluation_status(model, r),
            })

for key in [('Prompt-ZS', 0), ('Prompt-2shot', 2), ('Prompt-4shot', 4), ('Prompt-8shot', 8)]:
    r = results.get(key)
    if r:
        result_rows.append({
            'model': key[0],
            'N': key[1],
            'accuracy_mean': metric_or_nan(r, 'accuracy_mean', 'mean'),
            'accuracy_std': metric_or_nan(r, 'accuracy_std', 'std'),
            'macro_f1_mean': metric_or_nan(r, 'macro_f1_mean'),
            'macro_f1_std': metric_or_nan(r, 'macro_f1_std'),
            'train_time_mean': metric_or_nan(r, 'train_time_mean'),
            'inference_time_per_example': metric_or_nan(r, 'inference_time_per_example_mean'),
            'accuracy_parsed_mean': metric_or_nan(r, 'accuracy_parsed_mean'),
            'parse_rate_mean': metric_or_nan(r, 'parse_rate_mean'),
            'unparsed_mean': metric_or_nan(r, 'unparsed_mean'),
            'n_eval_mean': metric_or_nan(r, 'n_eval_mean'),
            'evaluation_status': evaluation_status(key[0], r),
        })

print(f"{'Model':<20} {'N':>6} {'Acc':>8} {'Std':>8} {'Macro-F1':>10} {'Train s':>10} {'Infer/ex':>10} {'Status':>12}")
print("-" * 98)
for row in result_rows:
    print(
        f"{row['model']:<20} {row['N']:>6} "
        f"{format_number(row['accuracy_mean']):>8} {format_number(row['accuracy_std']):>8} "
        f"{format_number(row['macro_f1_mean']):>10} "
        f"{format_number(row['train_time_mean'], 2):>10} "
        f"{format_number(row['inference_time_per_example'], 6):>10} "
        f"{row['evaluation_status']:>12}"
    )

csv_path = output_dir / "rq4_results_table.csv"
md_path = output_dir / "rq4_results_table.md"
fieldnames = [
    'model', 'N', 'accuracy_mean', 'accuracy_std', 'macro_f1_mean', 'macro_f1_std',
    'train_time_mean', 'inference_time_per_example',
    'accuracy_parsed_mean', 'parse_rate_mean', 'unparsed_mean', 'n_eval_mean',
    'evaluation_status',
]

with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(result_rows)

lines = [
    "| Model | N | Accuracy mean | Accuracy std | Macro-F1 mean | Macro-F1 std | Train time mean | Inference time/example | n eval | Parse rate | Status |",
    "|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|",
]
for row in result_rows:
    lines.append(
        f"| {row['model']} | {row['N']} | {format_number(row['accuracy_mean'])} | "
        f"{format_number(row['accuracy_std'])} | {format_number(row['macro_f1_mean'])} | "
        f"{format_number(row['macro_f1_std'])} | {format_number(row['train_time_mean'], 2)} | "
        f"{format_number(row['inference_time_per_example'], 6)} | {format_number(row['n_eval_mean'], 0)} | "
        f"{format_number(row['parse_rate_mean'])} | {row['evaluation_status']} |"
    )
md_path.write_text("\n".join(lines), encoding="utf-8")

print(f"\nSaved result table to {csv_path} and {md_path}")


---
## Part 7: Error Analysis

The cell below runs a small, explicit error analysis at n=200. It uses a random balanced test subset for readability and runtime, then exports concrete examples with review snippet, gold label, LSTM prediction, BERT prediction, and prompting prediction.


In [ ]:
# -- Error analysis ------------------------------------------------------
from pathlib import Path
import csv
ERROR_N = 200
ERROR_EVAL_N = 200
PROMPT_ERROR_EVAL_N = ERROR_EVAL_N

def clean_snippet(text, max_chars=350):
    text = re.sub(r"\s+", " ", text).strip()
    return text[:max_chars] + ("..." if len(text) > max_chars else "")

def label_text(label):
    if label == 1:
        return "Positive"
    if label == 0:
        return "Negative"
    return "Unparsed"

def train_lstm_for_errors(train_subset, eval_dataset, vocab, epochs=3, lr=1e-3):
    """Train one small LSTM model and return predictions for error analysis."""
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]
    train_ds = SentimentDataset(train_texts, train_labels, vocab)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

    model = LSTMClassifier(len(vocab)).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

    model.eval()
    preds, labels = [], []
    loader = DataLoader(eval_dataset, batch_size=128)
    with torch.no_grad():
        for bx, by in loader:
            bx = bx.to(device)
            preds.extend(model(bx).argmax(1).cpu().numpy().tolist())
            labels.extend(by.numpy().tolist())
    return preds, labels

def train_bert_for_errors(train_subset, eval_examples, epochs=1):
    """Train one small DistilBERT model and return predictions for error analysis."""
    train_texts = [ex['text'] for ex in train_subset]
    train_labels = [ex['label'] for ex in train_subset]
    eval_texts = [ex['text'] for ex in eval_examples]
    eval_labels = [ex['label'] for ex in eval_examples]

    train_enc = bert_tokenizer(train_texts, truncation=True, padding="max_length",
                               max_length=256, return_tensors="pt")
    train_enc['labels'] = torch.tensor(train_labels)
    eval_enc = bert_tokenizer(eval_texts, truncation=True, padding="max_length",
                              max_length=256, return_tensors="pt")
    eval_enc['labels'] = torch.tensor(eval_labels)

    model = AutoModelForSequenceClassification.from_pretrained(BERT_MODEL, num_labels=2)
    args = TrainingArguments(
        output_dir="./tmp_rq4_error_bert",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        save_strategy="no",
        logging_steps=9999,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=args, train_dataset=SimpleDataset(train_enc))
    trainer.train()
    pred_output = trainer.predict(SimpleDataset(eval_enc))
    return np.argmax(pred_output.predictions, axis=-1).tolist(), eval_labels

def show_examples(title, indices, preds_by_model, labels, texts, max_examples=5):
    print("\n" + title)
    print("-" * len(title))
    if not indices:
        print("No examples found in this subset.")
        return []
    shown = []
    for i in indices[:max_examples]:
        pred_text = ", ".join(f"{name}: {label_text(preds[i])}" for name, preds in preds_by_model.items())
        print(f"True: {label_text(labels[i])} | {pred_text}")
        print(clean_snippet(texts[i]))
        print()
        shown.append((title, i))
    return shown

error_train = get_subset(dataset, ERROR_N, seed=123)
error_eval_examples = list(get_balanced_split_subset(dataset['test'], ERROR_EVAL_N, seed=777))
error_eval_texts = [ex['text'] for ex in error_eval_examples]
error_eval_labels = [ex['label'] for ex in error_eval_examples]
error_eval_ds = SentimentDataset(error_eval_texts, error_eval_labels, vocab)

lstm_preds, labels = train_lstm_for_errors(error_train, error_eval_ds, vocab)

bert_preds = None
try:
    bert_preds, _ = train_bert_for_errors(error_train, error_eval_examples, epochs=1)
except Exception as e:
    print(f"BERT error-analysis run skipped because: {e}")

prompt_preds = None
try:
    rng = np.random.RandomState(2026)
    pos = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 1]
    neg = [i for i, ex in enumerate(dataset['train']) if ex['label'] == 0]
    ex_idx = list(rng.choice(pos, 4, replace=False)) + list(rng.choice(neg, 4, replace=False))
    rng.shuffle(ex_idx)
    prompt_examples = [(dataset['train'][i]['text'], dataset['train'][i]['label']) for i in ex_idx]
    prompt_preds = [
        classify_with_prompt(text, prompt_examples)
        for text in error_eval_texts[:PROMPT_ERROR_EVAL_N]
    ]
except Exception as e:
    print(f"Prompting error-analysis run skipped because: {e}")

preds_by_model = {'LSTM': lstm_preds}
if bert_preds is not None:
    preds_by_model['BERT'] = bert_preds

selected_examples = []
if bert_preds is not None:
    bert_right_lstm_wrong = [
        i for i, y in enumerate(labels)
        if bert_preds[i] == y and lstm_preds[i] != y
    ]
    selected_examples.extend(show_examples("BERT correct, LSTM wrong", bert_right_lstm_wrong, preds_by_model,
                                           labels, error_eval_texts))

if prompt_preds is not None and bert_preds is not None:
    prompt_right_models_wrong = [
        i for i, y in enumerate(labels[:PROMPT_ERROR_EVAL_N])
        if prompt_preds[i] == y and lstm_preds[i] != y and bert_preds[i] != y
    ]
    prompt_models = {'LSTM': lstm_preds[:PROMPT_ERROR_EVAL_N],
                     'BERT': bert_preds[:PROMPT_ERROR_EVAL_N],
                     'Prompt': prompt_preds}
    selected_examples.extend(show_examples("Prompt correct, trained models wrong", prompt_right_models_wrong,
                                           prompt_models, labels[:PROMPT_ERROR_EVAL_N],
                                           error_eval_texts[:PROMPT_ERROR_EVAL_N]))

all_wrong = [
    i for i, y in enumerate(labels)
    if lstm_preds[i] != y and (bert_preds is None or bert_preds[i] != y)
]
selected_examples.extend(show_examples("Hard examples: all available trained models wrong", all_wrong,
                                       preds_by_model, labels, error_eval_texts))

output_dir = Path("rq4_outputs")
output_dir.mkdir(exist_ok=True)
error_rows = []
seen = set()
for category, i in selected_examples:
    if (category, i) in seen:
        continue
    seen.add((category, i))
    row = {
        'category': category,
        'review_snippet': clean_snippet(error_eval_texts[i], max_chars=500),
        'gold_label': label_text(labels[i]),
        'lstm_prediction': label_text(lstm_preds[i]),
        'bert_prediction': label_text(bert_preds[i]) if bert_preds is not None else "",
    }
    if prompt_preds is not None and i < len(prompt_preds):
        row['prompt_prediction'] = label_text(prompt_preds[i])
    else:
        row['prompt_prediction'] = ""
    error_rows.append(row)

error_csv_path = output_dir / "rq4_error_analysis_examples.csv"
error_md_path = output_dir / "rq4_error_analysis_examples.md"
with error_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        'category', 'review_snippet', 'gold_label',
        'lstm_prediction', 'bert_prediction', 'prompt_prediction'
    ])
    writer.writeheader()
    writer.writerows(error_rows)

md_lines = ["| Category | Gold | LSTM | BERT | Prompt | Review snippet |",
            "|---|---|---|---|---|---|"]
for row in error_rows:
    snippet = row['review_snippet'].replace("|", "\\|")
    md_lines.append(
        f"| {row['category']} | {row['gold_label']} | {row['lstm_prediction']} | "
        f"{row['bert_prediction']} | {row['prompt_prediction']} | {snippet} |"
    )
error_md_path.write_text("\n".join(md_lines), encoding="utf-8")

category_counts = {}
for row in error_rows:
    category_counts[row['category']] = category_counts.get(row['category'], 0) + 1
summary_path = output_dir / "rq4_error_analysis_summary.md"
summary_lines = ["| Category | Exported examples |", "|---|---:|"]
for category in [
    "BERT correct, LSTM wrong",
    "Prompt correct, trained models wrong",
    "Hard examples: all available trained models wrong",
]:
    summary_lines.append(f"| {category} | {category_counts.get(category, 0)} |")
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

print(f"Saved error-analysis examples to {error_csv_path} and {error_md_path}")
print("Visible patterns to check: negation, mixed sentiment, sarcasm, and reviews where plot summary dominates opinion.")


---
## Part 8: Short Analysis

The notebook keeps only a compact computed summary here. More detailed experiment notes are saved in `RQ4_experiment_details.md`.


In [ ]:
# -- Short additional analysis and external notes -----------------------
from pathlib import Path

RQ3_TEXTCNN_BEST_ACC = 0.8710  # from RQ3: TextCNN at seq_len=512
RQ3_TEXTCNN_BEST_F1 = 0.8710

def acc(approach, n):
    row = results.get((approach, n))
    return result_value((approach, n)) if row else np.nan

def metric(approach, n, field):
    key = (approach, n)
    return result_value(key, field) if key in results else np.nan

reference_n = 200 if 200 in TRAIN_SIZES else TRAIN_SIZES[0]
gap_glove = acc('LSTM+GloVe', reference_n) - acc('LSTM', reference_n)
gap_arch = acc('BiLSTM+Attn', reference_n) - acc('LSTM+GloVe', reference_n)
gap_bert = acc('BERT', reference_n) - acc('BiLSTM+Attn', reference_n)

gap_rows = []
for n in TRAIN_SIZES:
    gap_rows.append({
        'N': n,
        'representation_gap': acc('LSTM+GloVe', n) - acc('LSTM', n),
        'architecture_gap': acc('BiLSTM+Attn', n) - acc('LSTM+GloVe', n),
        'contextual_gap': acc('BERT', n) - acc('BiLSTM+Attn', n),
    })

trained_candidates = []
for approach in ['LSTM', 'LSTM+GloVe', 'BiLSTM+Attn', 'BERT']:
    value = acc(approach, reference_n)
    if not np.isnan(value):
        trained_candidates.append((approach, value))
best_trained = max(trained_candidates, key=lambda x: x[1]) if trained_candidates else None

prompt_candidates = []
for key in [('Prompt-ZS', 0), ('Prompt-2shot', 2), ('Prompt-4shot', 4), ('Prompt-8shot', 8)]:
    if key in results:
        prompt_candidates.append((key[0], result_value(key)))
best_prompt = max(prompt_candidates, key=lambda x: x[1]) if prompt_candidates else None

bert_vs_prompt = "not observed"
if best_prompt:
    for n in TRAIN_SIZES:
        if acc('BERT', n) > best_prompt[1]:
            bert_vs_prompt = n
            break

best_bilstm_full = acc('BiLSTM+Attn', 25000)
textcnn_gap = best_bilstm_full - RQ3_TEXTCNN_BEST_ACC

stale_prompt_rows = []
for row in result_rows:
    if row['model'].startswith('Prompt') and row.get('evaluation_status', '').startswith('STALE_SUBSET_RUN'):
        stale_prompt_rows.append(row['model'])

bert_eval_note = "full IMDB test split"
if BERT_EVAL_MAX_EXAMPLES is not None:
    bert_eval_note = f"random balanced test subset with n={BERT_EVAL_MAX_EXAMPLES}"
prompt_eval_note = "full IMDB test split" if PROMPT_EVAL_N is None else f"random balanced test subset with n={PROMPT_EVAL_N}"

print(f"Reference size: n={reference_n}")
print(f"GloVe effect: {gap_glove:+.4f}")
print(f"BiLSTM+attention effect: {gap_arch:+.4f}")
print(f"BERT contextual pretraining gap: {gap_bert:+.4f}")
if best_trained:
    print(f"Best trained model at n={reference_n}: {best_trained[0]} ({best_trained[1]:.4f})")
if best_prompt:
    print(f"Best prompting baseline: {best_prompt[0]} ({best_prompt[1]:.4f})")
print(f"BERT overtakes best prompting baseline at n = {bert_vs_prompt}")
print(f"BiLSTM+GloVe+Attn full-data vs RQ3 TextCNN: {best_bilstm_full:.4f} vs {RQ3_TEXTCNN_BEST_ACC:.4f} ({textcnn_gap:+.4f})")
if stale_prompt_rows:
    print(f"Warning: stale non-full-test prompting rows detected: {', '.join(stale_prompt_rows)}")

details = []
details.append("# RQ4 Experiment Details\n")
details.append("## What was completed\n")
details.append("- Compared LSTM, LSTM+GloVe, BiLSTM+GloVe+Attention, BERT fine-tuning, and prompting.\n")
details.append("- Stored accuracy, macro-F1, train time, inference time per example, parse rate, and unparsed prompting counts structurally.\n")
details.append("- BERT and prompting are configured to use the full IMDB test split by default.\n")
details.append("- Error analysis exports concrete examples and category counts for direct report use.\n")
details.append("\n## Experiment setup\n")
details.append(f"- Training sizes configured in the notebook: {TRAIN_SIZES}.\n")
details.append(f"- Number of seeds configured in the notebook: {N_SEEDS}.\n")
details.append("- Training subsets are balanced by the provided get_subset helper.\n")
details.append("- LSTM-family models evaluate on the full IMDB test split.\n")
details.append(f"- BERT evaluation: {bert_eval_note}.\n")
details.append(f"- Prompting evaluation: {prompt_eval_note}.\n")
if stale_prompt_rows:
    details.append(f"- Warning: Prompting exports for {', '.join(stale_prompt_rows)} are stale subset runs. Rerun the Prompting, Plot, Results Table, and Final Analysis cells after setting PROMPT_EVAL_N=None to regenerate full-test values.\n")

details.append("\n## Result table\n")
details.append("| Model | N | Accuracy mean | Accuracy std | Macro-F1 mean | Macro-F1 std | Train time mean | Inference time/example | n eval | Parse rate | Status |\n")
details.append("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---|\n")
for row in result_rows:
    details.append(
        f"| {row['model']} | {row['N']} | {format_number(row['accuracy_mean'])} | "
        f"{format_number(row['accuracy_std'])} | {format_number(row['macro_f1_mean'])} | "
        f"{format_number(row['macro_f1_std'])} | {format_number(row['train_time_mean'], 2)} | "
        f"{format_number(row['inference_time_per_example'], 6)} | {format_number(row['n_eval_mean'], 0)} | "
        f"{format_number(row['parse_rate_mean'])} | {row.get('evaluation_status', '')} |\n"
    )

details.append("\n## Gap decomposition by size\n")
details.append("| N | Representation: LSTM -> GloVe | Architecture: GloVe -> BiLSTM+Attn | Contextual: BiLSTM+Attn -> BERT |\n")
details.append("|---:|---:|---:|---:|\n")
for row in gap_rows:
    details.append(
        f"| {row['N']} | {row['representation_gap']:+.4f} | "
        f"{row['architecture_gap']:+.4f} | {row['contextual_gap']:+.4f} |\n"
    )

details.append("\n## Crossover and findings\n")
details.append(f"- Reference size: n={reference_n}.\n")
details.append(f"- At n={reference_n}, representation effect: {gap_glove:+.4f}.\n")
details.append(f"- At n={reference_n}, architecture/pooling effect: {gap_arch:+.4f}.\n")
details.append(f"- At n={reference_n}, contextual pretraining gap: {gap_bert:+.4f}.\n")
details.append(f"- BERT overtakes LSTM at n={first_overtake('BERT', 'LSTM')}.\n")
details.append(f"- BERT overtakes LSTM+GloVe at n={first_overtake('BERT', 'LSTM+GloVe')}.\n")
details.append(f"- BERT overtakes BiLSTM+GloVe+Attn at n={first_overtake('BERT', 'BiLSTM+Attn')}.\n")
if best_prompt:
    details.append(f"- Best prompting baseline: {best_prompt[0]} ({best_prompt[1]:.4f}).\n")
    details.append(f"- BERT overtakes the best prompting baseline at n={bert_vs_prompt}.\n")
if best_trained:
    details.append(f"- Best trained model at n={reference_n}: {best_trained[0]} ({best_trained[1]:.4f}).\n")
details.append(f"- BiLSTM+GloVe+Attention at full data: {best_bilstm_full:.4f}; RQ3 TextCNN best: {RQ3_TEXTCNN_BEST_ACC:.4f}; gap: {textcnn_gap:+.4f}.\n")

details.append("\n## Practical recommendation\n")
details.append("- With 0 labels, prompting is the only zero-training option, but GPT-2 is a weak local baseline and should be replaced by an instruction-tuned LLM for a realistic deployment.\n")
details.append("- With 50-200 labels, compare DistilBERT and BiLSTM+GloVe+Attention; in the current run BiLSTM+GloVe+Attention is strongest at n=200, while BERT catches up as data grows.\n")
details.append("- With 1000+ labels, fine-tuned DistilBERT/BERT is the strongest candidate if compute is available.\n")
details.append("- If inference cost matters, LSTM-family models are much cheaper per example than prompting.\n")

details.append("\n## References to cite in the report\n")
details.append("- Devlin et al. (2019), BERT.\n")
details.append("- Brown et al. (2020), Language Models are Few-Shot Learners.\n")
details.append("- Hochreiter and Schmidhuber (1997), LSTM.\n")
details.append("- Pennington et al. (2014), GloVe.\n")
details.append("- Kim (2014), TextCNN.\n")

details.append("\n## Limitations\n")
details.append("- GPT-2 is not instruction-tuned, so prompting results should be interpreted as a weak local baseline.\n")
details.append("- If any evaluation status is marked STALE_SUBSET_RUN, those rows must be regenerated before using them as full-test results.\n")
details.append("- If BERT_EVAL_MAX_EXAMPLES or PROMPT_EVAL_N is set for runtime, report the configured random balanced subset size instead of claiming full-test evaluation.\n")

details_path = Path("RQ4_experiment_details.md")
details_path.write_text("".join(details), encoding="utf-8")
print(f"Saved detailed experiment notes to {details_path}")


---
## Summary

Use the generated files in rq4_outputs/ and RQ4_experiment_details.md for the report.

| Finding | Evidence |
|---------|----------|
| Gap 1->2: representation effect | Computed in the final analysis as LSTM+GloVe minus LSTM for each training size. |
| Gap 2->3: architecture/pooling effect | Computed as BiLSTM+GloVe+Attention minus LSTM+GloVe. |
| Gap 3->4: contextual pretraining premium | Computed as BERT minus BiLSTM+GloVe+Attention. |
| BiLSTM+GloVe+Attention vs TextCNN from RQ3 | Compared against the RQ3 TextCNN best accuracy of 0.8710. |
| BERT vs few-shot prompting | The plot and final analysis report the first training size where BERT exceeds the best prompting baseline. |
| Zero-shot prompting accuracy | Stored in the Prompt-ZS row of the results table, with parse rate and unparsed count. |
| Best approach at n=200 | Printed by the final analysis cell. |
| Training time and inference cost | Stored in train_time_mean and inference_time_per_example. |

### Practical Recommendation

Use prompting when no labeled data is available, preferably with a stronger instruction-tuned model than GPT-2. With small labeled sets, compare BiLSTM+GloVe+Attention and DistilBERT because the current result can be close at n=200. With 1000+ labels, fine-tuned BERT/DistilBERT is usually the strongest accuracy choice if compute is available. For low-latency or low-cost inference, LSTM-family models are much cheaper per example than prompting.

### Key Takeaway

Pretraining is not a single jump. GloVe tests representation quality, BiLSTM+attention tests architecture and pooling, and BERT adds contextual pretraining. The result table and gap decomposition show which part of that ladder matters most at each data budget.
